In [ ]:
#Sincroniza Parquet RAW NRT (DGT, NASA NRT, Weather) a Tablas Delta
from tfm_mobility.processors.bronze_processor import BronzeProcessor

print("[BRONZE REAL-TIME] Iniciando promoción de fuentes NRT a Tablas Delta...")

# Instanciamos el procesador con la sesión activa de Spark
processor = BronzeProcessor(spark)

# Ejecutamos la promoción exclusiva de Real-Time
processor.promote_realtime_to_bronze()

print("[BRONZE REAL-TIME OK] Proceso finalizado. Tablas 'bronze_nasa_nrt', 'bronze_weather' y 'bronze_dgt_traffic' actualizadas.")

In [ ]:
# ==============================================================================
# NOTEBOOK: 03.1_data_quality_audit
# DESCRIPCIÓN: Auditoría completa de calidad, métricas y exploración de Bronze
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, DoubleType, FloatType, IntegerType, LongType

def run_data_quality_audit():
    # Lista de tablas Bronze registradas en el catálogo
    tables = [t.name for t in spark.catalog.listTables() if t.name.startswith("bronze_")]
    
    if not tables:
        print("⚠️ No se encontraron tablas 'bronze_' en el catálogo. Revisa 'show tables'.")
        return

    print("================================================================================")
    print(f"🔍 AUDITORÍA DE CALIDAD Y EXPLORACIÓN DE DATOS BRONZE ({len(tables)} tablas)")
    print("================================================================================\n")

    for table_name in sorted(tables):
        print(f"--------------------------------------------------------------------------------")
        print(f"📊 AUDITANDO TABLA DELTA: {table_name.upper()}")
        print(f"--------------------------------------------------------------------------------")
        
        try:
            df = spark.table(table_name)
            total_records = df.count()
            print(f"🔹 Nº Total de Registros: {total_records:,}")

            if total_records == 0:
                print("⚠️ La tabla está vacía.\n")
                continue

            # 1. BÚSQUEDA DE MARCAS TEMPORALES / DATES DE INGESTA
            date_cols = [c for c, t in df.dtypes if any(term in c.lower() for term in ["time", "date", "fecha", "ingest", "timestamp"])]
            if date_cols:
                print("\n📅 Rango de Fechas / Marcas Temporales de Ingesta:")
                for d_col in date_cols[:3]: # Evaluamos las 3 primeras columnas temporales
                    agg_dates = df.select(F.min(d_col).alias("min_date"), F.max(d_col).alias("max_date"), F.countDistinct(d_col).alias("dist_dates")).collect()[0]
                    print(f"   • Columna '{d_col}':")
                    print(f"     - Distintas fechas/timestamps: {agg_dates['dist_dates']:,}")
                    print(f"     - Rango: Desde [{agg_dates['min_date']}] Hasta [{agg_dates['max_date']}]")
            else:
                print("\n📅 Marcas temporales: No se detectaron columnas explícitas de fecha/hora.")

            # 2. CONTEO DE VALORES ÚNICOS POR COLUMNA
            print("\n🔑 Valores Únicos (Count Distinct) y Tipo de Dato por Columna:")
            distinct_exprs = [F.countDistinct(F.col(c)).alias(c) for c in df.columns]
            distinct_counts = df.select(distinct_exprs).collect()[0].asDict()
            
            for col_name, dtype in df.dtypes:
                u_cnt = distinct_counts[col_name]
                print(f"   • {col_name} ({dtype}): {u_cnt:,} valores únicos")

            # 3. ESTADÍSTICAS NUMÉRICAS (Mínimo, Máximo, Media, Moda)
            numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (NumericType, DoubleType, FloatType, IntegerType, LongType))]
            
            if numeric_cols:
                print("\n📈 Estadísticas para Columnas Numéricas (Mínimo, Máximo, Media, Moda):")
                for num_col in numeric_cols:
                    # Mínimo, Máximo y Media
                    stats = df.select(
                        F.min(num_col).alias("min"),
                        F.max(num_col).alias("max"),
                        F.avg(num_col).alias("mean")
                    ).collect()[0]

                    # Cálculo de la Moda (valor más frecuente)
                    mode_df = df.groupBy(num_col).count().orderBy(F.col("count").desc()).limit(1).collect()
                    mode_val = mode_df[0][num_col] if mode_df and mode_df[0][num_col] is not None else "N/A"
                    
                    mean_val = round(stats["mean"], 4) if stats["mean"] is not None else "N/A"
                    print(f"   • {num_col}:")
                    print(f"     - Mínimo: {stats['min']}")
                    print(f"     - Máximo: {stats['max']}")
                    print(f"     - Media : {mean_val}")
                    print(f"     - Moda  : {mode_val}")
            else:
                print("\n📈 Estadísticas numéricas: No se detectaron columnas numéricas continuas.")

            # 4. MUESTRA DE REGISTROS DE EJEMPLO
            print("\n📄 Muestra de 3 Registros de Ejemplo:")
            df.show(3, truncate=80)
            print("\n")

        except Exception as e:
            print(f"❌ Error auditando la tabla {table_name}: {e}\n")

# Ejecución de la auditoría
run_data_quality_audit()